In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install torch torchvision torchaudio --quiet
!pip install opencv-python tqdm scikit-image pandas matplotlib --quiet


In [ ]:
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
import os
import glob
from skimage.metrics import mean_squared_error
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
class ConvBNReLU(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = ConvBNReLU(channels, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.bn = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.bn(x)
        x = x + residual
        x = self.relu(x)
        return x

class Encoder(nn.Module):
    def __init__(self, in_channels=4):
        super().__init__()
        self.conv1 = ConvBNReLU(in_channels, 32, kernel_size=3, stride=1, padding=1)
        self.down1 = ConvBNReLU(32, 64, kernel_size=3, stride=2, padding=1)
        self.down2 = ConvBNReLU(64, 128, kernel_size=3, stride=2, padding=1)
        self.down3 = ConvBNReLU(128, 256, kernel_size=3, stride=2, padding=1)
        self.down4 = ConvBNReLU(256, 512, kernel_size=3, stride=2, padding=1)

        self.res1 = ResidualBlock(64)
        self.res2 = ResidualBlock(128)
        self.res3 = ResidualBlock(256)
        self.res4 = ResidualBlock(512)

    def forward(self, x):
        features = {}
        x = self.conv1(x)
        x = self.down1(x)
        x = self.res1(x)
        features['level1'] = x
        x = self.down2(x)
        x = self.res2(x)
        features['level2'] = x
        x = self.down3(x)
        x = self.res3(x)
        features['level3'] = x
        x = self.down4(x)
        x = self.res4(x)
        features['level4'] = x
        return features

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.up4 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBNReLU(512, 256)
        )
        self.conv4 = ResidualBlock(256)

        self.up3 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBNReLU(256, 128)
        )
        self.conv3 = ResidualBlock(128)

        self.up2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBNReLU(128, 64)
        )
        self.conv2 = ResidualBlock(64)

        self.up1 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBNReLU(64, 32)
        )
        self.conv1 = ResidualBlock(32)

        self.alpha_out = nn.Sequential(
            ConvBNReLU(32, 16),
            nn.Conv2d(16, 1, 3, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, encoder_features):
        x = encoder_features['level4']
        x = self.up4(x)
        x = x + encoder_features['level3']
        x = self.conv4(x)
        x = self.up3(x)
        x = x + encoder_features['level2']
        x = self.conv3(x)
        x = self.up2(x)
        x = x + encoder_features['level1']
        x = self.conv2(x)
        x = self.up1(x)
        x = self.conv1(x)
        alpha = self.alpha_out(x)
        return alpha

class SemanticRefineNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder(in_channels=4)
        self.decoder = Decoder()

    def forward(self, rgb, base_alpha):
        x = torch.cat([rgb, base_alpha], dim=1)
        features = self.encoder(x)
        semantic_alpha = self.decoder(features)
        return semantic_alpha

In [ ]:
class OriginalRVM:
    def __init__(self, model_type="resnet50"):  #   mobilenetv3, resnet50
        self.model = torch.hub.load(
            "PeterL1n/RobustVideoMatting",
            model_type,
            pretrained=True
        )
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = self.model.to(self.device)
        self.model.eval()

    def predict(self, image_bgr):
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        image_tensor = torch.from_numpy(image_rgb).permute(2, 0, 1).unsqueeze(0)
        image_tensor = image_tensor.float() / 255.0
        image_tensor = image_tensor.to(self.device)

        with torch.no_grad():
            fgr, pha, *rec = self.model(image_tensor, None, None)

        alpha = pha.squeeze().cpu().numpy()
        alpha = np.clip(alpha, 0, 1)
        return alpha

In [ ]:
def load_your_model(model_path, device):
    model = SemanticRefineNet().to(device)
    checkpoint = torch.load(model_path, map_location=device)

    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        epoch = checkpoint.get('epoch', 'unknown')
        val_loss = checkpoint.get('val_loss', 'unknown')
    else:
        model.load_state_dict(checkpoint)

    model.eval()
    return model


In [ ]:
def refine_alpha(rvm_alpha, semantic_alpha, delta=0.7, transition_range=(0.05, 0.95)):
    low, high = transition_range
    weight = np.zeros_like(rvm_alpha)
    transition_mask = (rvm_alpha > low) & (rvm_alpha < high)
    weight[transition_mask] = delta
    weight = cv2.GaussianBlur(weight, (5, 5), 1.0)
    refined = (1 - weight) * rvm_alpha + weight * semantic_alpha
    refined = np.clip(refined, 0, 1)
    return refined


def get_semantic_alpha(model, image_bgr, device, target_size=(512, 512)):
    rvm_model_tmp = OriginalRVM()
    base_alpha = rvm_model_tmp.predict(image_bgr)

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_resized = cv2.resize(image_rgb, target_size)
    base_resized = cv2.resize(base_alpha, target_size)

    image_tensor = torch.from_numpy(image_resized).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    base_tensor = torch.from_numpy(base_resized).unsqueeze(0).unsqueeze(0).float()

    image_tensor = image_tensor.to(device)
    base_tensor = base_tensor.to(device)

    with torch.no_grad():
        semantic_alpha = model(image_tensor, base_tensor)
        semantic_alpha = semantic_alpha.squeeze().cpu().numpy()

    h, w = image_bgr.shape[:2]
    semantic_alpha = cv2.resize(semantic_alpha, (w, h))
    semantic_alpha = np.clip(semantic_alpha, 0, 1)

    return semantic_alpha, base_alpha

In [ ]:
def compute_metrics(pred, gt):
    pred = np.clip(pred, 0, 1)
    gt = np.clip(gt, 0, 1)
    mse = mean_squared_error(gt, pred)
    sad = np.sum(np.abs(pred - gt))
    return mse, sad


def evaluate_testset_with_precomputed(testset_root, your_model, device,
                                       delta=0.7, target_size=(512, 512),
                                       base_alpha_dir_name="base_alpha"):
    
    image_dir = os.path.join(testset_root, 'original_image')
    if not os.path.exists(image_dir):
        image_dir = os.path.join(testset_root, 'blurred_image')

    mask_dir = os.path.join(testset_root, 'mask')
    base_alpha_dir = os.path.join(testset_root, base_alpha_dir_name)

    if not os.path.exists(base_alpha_dir):
        return evaluate_testset(testset_root, None, your_model, device, delta, target_size)

    valid_pairs = []
    for ext in ['*.jpg', '*.jpeg', '*.png']:
        for img_path in glob.glob(os.path.join(image_dir, ext)):
            name = os.path.basename(img_path).split('.')[0]
            mask_path = os.path.join(mask_dir, f"{name}.png")
            base_path = os.path.join(base_alpha_dir, f"{name}.png")

            if not os.path.exists(mask_path):
                mask_path = os.path.join(mask_dir, f"{name}.jpg")
            if not os.path.exists(base_path):
                base_path = os.path.join(base_alpha_dir, f"{name}.jpg")

            if os.path.exists(mask_path) and os.path.exists(base_path):
                valid_pairs.append((img_path, mask_path, base_path, name))

    valid_pairs = sorted(valid_pairs)
    print(f"\n Find {len(valid_pairs)} images")
    print(f"   pre-generate base_alpha: {base_alpha_dir}")
    print(f"   🔧 Refinement δ = {delta}")
    print("=" * 60)

    results = []

    for img_path, mask_path, base_path, name in tqdm(valid_pairs, desc="Evaluating"):
        image = cv2.imread(img_path)
        gt_alpha = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_alpha = gt_alpha.astype(np.float32) / 255.0

        base_alpha = cv2.imread(base_path, cv2.IMREAD_GRAYSCALE)
        base_alpha = base_alpha.astype(np.float32) / 255.0

        if image is None or gt_alpha is None or base_alpha is None:
            print(f"   Skip {name}")
            continue

        h, w = image.shape[:2]
        if base_alpha.shape[:2] != (h, w):
            base_alpha = cv2.resize(base_alpha, (w, h))

        rvm_alpha = base_alpha

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_resized = cv2.resize(image_rgb, target_size)
        base_resized = cv2.resize(rvm_alpha, target_size)

        img_tensor = torch.from_numpy(image_resized).permute(2, 0, 1).unsqueeze(0).float() / 255.0
        base_tensor = torch.from_numpy(base_resized).unsqueeze(0).unsqueeze(0).float()

        img_tensor = img_tensor.to(device)
        base_tensor = base_tensor.to(device)

        with torch.no_grad():
            semantic_alpha = your_model(img_tensor, base_tensor)
            semantic_alpha = semantic_alpha.squeeze().cpu().numpy()
            semantic_alpha = cv2.resize(semantic_alpha, (w, h))
            semantic_alpha = np.clip(semantic_alpha, 0, 1)

        refined_alpha = refine_alpha(rvm_alpha, semantic_alpha, delta=delta)

        rvm_mse, rvm_sad = compute_metrics(rvm_alpha, gt_alpha)
        refined_mse, refined_sad = compute_metrics(refined_alpha, gt_alpha)

        transition_mask = (gt_alpha > 0.05) & (gt_alpha < 0.95)
        if transition_mask.any():
            rvm_trans_mse = mean_squared_error(gt_alpha[transition_mask], rvm_alpha[transition_mask])
            rvm_trans_sad = np.sum(np.abs(rvm_alpha[transition_mask] - gt_alpha[transition_mask]))
            refined_trans_mse = mean_squared_error(gt_alpha[transition_mask], refined_alpha[transition_mask])
            refined_trans_sad = np.sum(np.abs(refined_alpha[transition_mask] - gt_alpha[transition_mask]))
        else:
            rvm_trans_mse = rvm_trans_sad = refined_trans_mse = refined_trans_sad = 0

        results.append({
            'name': name,
            'rvm_mse': rvm_mse,
            'rvm_sad': rvm_sad,
            'rvm_trans_mse': rvm_trans_mse,
            'rvm_trans_sad': rvm_trans_sad,
            'refined_mse': refined_mse,
            'refined_sad': refined_sad,
            'refined_trans_mse': refined_trans_mse,
            'refined_trans_sad': refined_trans_sad,
            'improvement_mse': rvm_mse - refined_mse,
            'improvement_sad': rvm_sad - refined_sad,
            'improvement_percent_mse': ((rvm_mse - refined_mse) / rvm_mse * 100) if rvm_mse > 0 else 0,
            'improvement_percent_sad': ((rvm_sad - refined_sad) / rvm_sad * 100) if rvm_sad > 0 else 0,
        })

    df = pd.DataFrame(results)
    summary = {
        'total_images': len(df),
        'rvm_avg_mse': df['rvm_mse'].mean(),
        'rvm_avg_sad': df['rvm_sad'].mean(),
        'rvm_avg_trans_mse': df['rvm_trans_mse'].mean(),
        'rvm_avg_trans_sad': df['rvm_trans_sad'].mean(),
        'refined_avg_mse': df['refined_mse'].mean(),
        'refined_avg_sad': df['refined_sad'].mean(),
        'refined_avg_trans_mse': df['refined_trans_mse'].mean(),
        'refined_avg_trans_sad': df['refined_trans_sad'].mean(),
        'avg_improvement_mse': df['improvement_mse'].mean(),
        'avg_improvement_sad': df['improvement_sad'].mean(),
        'avg_improvement_percent_mse': df['improvement_percent_mse'].mean(),
        'avg_improvement_percent_sad': df['improvement_percent_sad'].mean(),
        'improved_count_mse': (df['improvement_mse'] > 0).sum(),
        'improved_count_sad': (df['improvement_sad'] > 0).sum(),
    }

    return df, summary


def evaluate_testset(testset_root, rvm_model, your_model, device, delta=0.7, target_size=(512, 512)):
    image_dir = os.path.join(testset_root, 'original_image')
    if not os.path.exists(image_dir):
        image_dir = os.path.join(testset_root, 'blurred_image')

    mask_dir = os.path.join(testset_root, 'mask')

    valid_pairs = []
    for ext in ['*.jpg', '*.jpeg', '*.png']:
        for img_path in glob.glob(os.path.join(image_dir, ext)):
            name = os.path.basename(img_path).split('.')[0]
            mask_path = os.path.join(mask_dir, f"{name}.png")
            if not os.path.exists(mask_path):
                mask_path = os.path.join(mask_dir, f"{name}.jpg")
            if os.path.exists(mask_path):
                valid_pairs.append((img_path, mask_path, name))

    valid_pairs = sorted(valid_pairs)
    print(f"\n Find {len(valid_pairs)} images")
    print(f"   🔧 Refinement δ = {delta}")
    print("=" * 60)

    results = []

    for img_path, mask_path, name in tqdm(valid_pairs, desc="Evaluating"):
        image = cv2.imread(img_path)
        gt_alpha = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_alpha = gt_alpha.astype(np.float32) / 255.0

        if image is None or gt_alpha is None:
            continue

        rvm_alpha = rvm_model.predict(image)

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_resized = cv2.resize(image_rgb, target_size)
        base_resized = cv2.resize(rvm_alpha, target_size)

        img_tensor = torch.from_numpy(image_resized).permute(2, 0, 1).unsqueeze(0).float() / 255.0
        base_tensor = torch.from_numpy(base_resized).unsqueeze(0).unsqueeze(0).float()

        img_tensor = img_tensor.to(device)
        base_tensor = base_tensor.to(device)

        with torch.no_grad():
            semantic_alpha = your_model(img_tensor, base_tensor)
            semantic_alpha = semantic_alpha.squeeze().cpu().numpy()

        h, w = image.shape[:2]
        semantic_alpha = cv2.resize(semantic_alpha, (w, h))
        semantic_alpha = np.clip(semantic_alpha, 0, 1)

        refined_alpha = refine_alpha(rvm_alpha, semantic_alpha, delta=delta)

        rvm_mse, rvm_sad = compute_metrics(rvm_alpha, gt_alpha)
        refined_mse, refined_sad = compute_metrics(refined_alpha, gt_alpha)

        transition_mask = (gt_alpha > 0.05) & (gt_alpha < 0.95)
        if transition_mask.any():
            rvm_trans_mse = mean_squared_error(gt_alpha[transition_mask], rvm_alpha[transition_mask])
            rvm_trans_sad = np.sum(np.abs(rvm_alpha[transition_mask] - gt_alpha[transition_mask]))
            refined_trans_mse = mean_squared_error(gt_alpha[transition_mask], refined_alpha[transition_mask])
            refined_trans_sad = np.sum(np.abs(refined_alpha[transition_mask] - gt_alpha[transition_mask]))
        else:
            rvm_trans_mse = rvm_trans_sad = refined_trans_mse = refined_trans_sad = 0

        results.append({
            'name': name,
            'rvm_mse': rvm_mse,
            'rvm_sad': rvm_sad,
            'rvm_trans_mse': rvm_trans_mse,
            'rvm_trans_sad': rvm_trans_sad,
            'refined_mse': refined_mse,
            'refined_sad': refined_sad,
            'refined_trans_mse': refined_trans_mse,
            'refined_trans_sad': refined_trans_sad,
            'improvement_mse': rvm_mse - refined_mse,
            'improvement_sad': rvm_sad - refined_sad,
            'improvement_percent_mse': ((rvm_mse - refined_mse) / rvm_mse * 100) if rvm_mse > 0 else 0,
            'improvement_percent_sad': ((rvm_sad - refined_sad) / rvm_sad * 100) if rvm_sad > 0 else 0,
        })

    df = pd.DataFrame(results)
    summary = {
        'total_images': len(df),
        'rvm_avg_mse': df['rvm_mse'].mean(),
        'rvm_avg_sad': df['rvm_sad'].mean(),
        'rvm_avg_trans_mse': df['rvm_trans_mse'].mean(),
        'rvm_avg_trans_sad': df['rvm_trans_sad'].mean(),
        'refined_avg_mse': df['refined_mse'].mean(),
        'refined_avg_sad': df['refined_sad'].mean(),
        'refined_avg_trans_mse': df['refined_trans_mse'].mean(),
        'refined_avg_trans_sad': df['refined_trans_sad'].mean(),
        'avg_improvement_mse': df['improvement_mse'].mean(),
        'avg_improvement_sad': df['improvement_sad'].mean(),
        'avg_improvement_percent_mse': df['improvement_percent_mse'].mean(),
        'avg_improvement_percent_sad': df['improvement_percent_sad'].mean(),
        'improved_count_mse': (df['improvement_mse'] > 0).sum(),
        'improved_count_sad': (df['improvement_sad'] > 0).sum(),
    }

    return df, summary

In [ ]:
def print_summary(summary):
    print("\n" + "=" * 60)
    print("Evaluation summary")
    print("=" * 60)
    print(f"\nTotal images: {summary['total_images']}")

    print("\n" + "-" * 40)
    print("MSE (Mean Squared Error)")
    print("-" * 40)
    print(f"   RVM:     {summary['rvm_avg_mse']:.6f}")
    print(f"   Semantic Refined: {summary['refined_avg_mse']:.6f}")
    print(f"   Improvement:           {summary['avg_improvement_mse']:.6f} ({summary['avg_improvement_percent_mse']:.2f}%)")
    print(f"   Improved count:     {summary['improved_count_mse']}/{summary['total_images']}")

    print("\n" + "-" * 40)
    print("SAD (Sum of Absolute Differences)")
    print("-" * 40)
    print(f"   RVM:     {summary['rvm_avg_sad']:.2f}")
    print(f"   Semantic Refined: {summary['refined_avg_sad']:.2f}")
    print(f"   Improvement:           {summary['avg_improvement_sad']:.2f} ({summary['avg_improvement_percent_sad']:.2f}%)")
    print(f"   Improved count:     {summary['improved_count_sad']}/{summary['total_images']}")

    print("\n" + "-" * 40)
    print("Transition region (0.05 < α < 0.95)")
    print("-" * 40)
    print(f"   RVM MSE:     {summary['rvm_avg_trans_mse']:.6f}")
    print(f"   Semantic Refined MSE: {summary['refined_avg_trans_mse']:.6f}")
    print(f"   RVM SAD:     {summary['rvm_avg_trans_sad']:.2f}")
    print(f"   Semantic Refined SAD: {summary['refined_avg_trans_sad']:.2f}")
    print("=" * 60)


In [ ]:
def plot_evaluation_results(df, summary, save_path=None):
    import matplotlib.pyplot as plt
    plt.rcParams['font.family'] = 'DejaVu Sans'

    if df is None or len(df) == 0:
        print("No plot")
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax1 = axes[0, 0]
    ax1.hist(df['rvm_mse'], bins=30, alpha=0.5, label=f'RVM (mean={summary["rvm_avg_mse"]:.5f})', color='blue')
    ax1.hist(df['refined_mse'], bins=30, alpha=0.5, label=f'Refined (mean={summary["refined_avg_mse"]:.5f})', color='red')
    ax1.set_xlabel('MSE')
    ax1.set_ylabel('Frequency')
    ax1.set_title('MSE Distribution Comparison')
    ax1.legend()

    ax2 = axes[0, 1]
    ax2.hist(df['rvm_sad'], bins=30, alpha=0.5, label=f'RVM (mean={summary["rvm_avg_sad"]:.1f})', color='blue')
    ax2.hist(df['refined_sad'], bins=30, alpha=0.5, label=f'Refined (mean={summary["refined_avg_sad"]:.1f})', color='red')
    ax2.set_xlabel('SAD')
    ax2.set_ylabel('Frequency')
    ax2.set_title('SAD Distribution Comparison')
    ax2.legend()

    ax3 = axes[1, 0]
    improvements = df['improvement_mse']
    ax3.scatter(range(len(improvements)), improvements, alpha=0.5, s=10)
    ax3.axhline(y=0, color='red', linestyle='--', label='No Improvement')
    ax3.set_xlabel('Image Index')
    ax3.set_ylabel('MSE Improvement')
    ax3.set_title('MSE Improvement per Image')
    ax3.legend()

    ax4 = axes[1, 1]
    improved_count = (df['improvement_mse'] > 0).sum()
    worsened_count = (df['improvement_mse'] < 0).sum()
    no_change_count = (df['improvement_mse'] == 0).sum()

    bars = ax4.bar(['Improved', 'Worsened', 'No Change'],
                   [improved_count, worsened_count, no_change_count],
                   color=['green', 'red', 'gray'])
    ax4.set_ylabel('Number of Images')
    ax4.set_title(f'Improvement Statistics: {improved_count}/{len(df)} images improved')

    for bar, v in zip(bars, [improved_count, worsened_count, no_change_count]):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(v), ha='center', fontweight='bold', fontsize=12)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)

    plt.show()

In [ ]:
def main():
    TESTSET_ROOT = "/content/drive/MyDrive/DATA/P3M-10k/validation/P3M-500-NP"
    MODEL_PATH = "/content/drive/MyDrive/RVM/semantic_refine_model/model_best.pth"
    SAVE_CSV = "/content/drive/MyDrive/RVM/full_evaluation_results2.csv"

    DELTA = 0.8                
    TARGET_SIZE = (512, 512)

    USE_PRECOMPUTED_BASE_ALPHA = True   
    BASE_ALPHA_DIR_NAME = "base_alpha" 

    print("=" * 60)
    print("Full Inference")
    print("=" * 60)
    print(f"testset root: {TESTSET_ROOT}")
    print(f"model path: {MODEL_PATH}")
    print(f"Refinement δ = {DELTA}")
    print(f"{'pre-generate base_alpha' if USE_PRECOMPUTED_BASE_ALPHA else 'RVM'}")
    print("=" * 60)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    your_model = load_your_model(MODEL_PATH, device)

    if USE_PRECOMPUTED_BASE_ALPHA:
        print("\npre-generate base_alpha")
        df, summary = evaluate_testset_with_precomputed(
            testset_root=TESTSET_ROOT,
            your_model=your_model,
            device=device,
            delta=DELTA,
            target_size=TARGET_SIZE,
            base_alpha_dir_name=BASE_ALPHA_DIR_NAME
        )
    else:
        print("\nRVM")
        rvm_model = OriginalRVM()
        df, summary = evaluate_testset(
            testset_root=TESTSET_ROOT,
            rvm_model=rvm_model,
            your_model=your_model,
            device=device,
            delta=DELTA,
            target_size=TARGET_SIZE
        )

    df.to_csv(SAVE_CSV, index=False)
    print(f"\nSaved at: {SAVE_CSV}")

    plot_save_path = os.path.join(os.path.dirname(SAVE_CSV), "evaluation_plot.png")
    plot_evaluation_results(df, summary, save_path=plot_save_path)

    print_summary(summary)

    return df, summary


if __name__ == "__main__":
    df, summary = main()

In [ ]:
import matplotlib.pyplot as plt

TESTSET_ROOT = "/content/drive/MyDrive/DATA/P3M-10k/validation/P3M-500-NP"
MODEL_PATH = "/content/drive/MyDrive/RVM/semantic_refine_model/model_best.pth"
DELTA = 0.8
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_PRECOMPUTED_BASE_ALPHA = True 

print("Load model...")
your_model = load_your_model(MODEL_PATH, DEVICE)

rvm_model = None
if not USE_PRECOMPUTED_BASE_ALPHA:
    rvm_model = OriginalRVM()


def composite_with_green_bg(image, alpha, bg_color=(0, 255, 0)):
    h, w = image.shape[:2]
    if alpha.shape[:2] != (h, w):
        alpha = cv2.resize(alpha, (w, h))

    bg = np.full((h, w, 3), bg_color, dtype=np.uint8)
    alpha_3ch = np.stack([alpha, alpha, alpha], axis=2)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    composite = (image_rgb * alpha_3ch + bg * (1 - alpha_3ch)).astype(np.uint8)
    return composite


def load_rvm_alpha_from_precomputed(testset_root, name, base_alpha_dir_name="base_alpha"):
    base_alpha_dir = os.path.join(testset_root, base_alpha_dir_name)
    base_path = os.path.join(base_alpha_dir, f"{name}.png")
    if not os.path.exists(base_path):
        base_path = os.path.join(base_alpha_dir, f"{name}.jpg")
    if os.path.exists(base_path):
        alpha = cv2.imread(base_path, cv2.IMREAD_GRAYSCALE)
        return alpha.astype(np.float32) / 255.0
    return None


def evaluate_single_image(name, testset_root, your_model, device, delta=0.7,
                          use_precomputed=True, rvm_model=None,
                          base_alpha_dir_name="base_alpha", save_path=None):
    image_dir = os.path.join(testset_root, 'original_image')
    if not os.path.exists(image_dir):
        image_dir = os.path.join(testset_root, 'blurred_image')
    mask_dir = os.path.join(testset_root, 'mask')

    img_path = os.path.join(image_dir, f"{name}.jpg")
    if not os.path.exists(img_path):
        img_path = os.path.join(image_dir, f"{name}.png")
    mask_path = os.path.join(mask_dir, f"{name}.png")

    if not os.path.exists(img_path):
        print(f"Not found: {name}")
        return None
    if not os.path.exists(mask_path):
        print(f"Not found GT mask: {name}")
        return None

    print(f"\nEvaluate image: {name}")

    image = cv2.imread(img_path)
    gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) / 255.0

    if use_precomputed:
        rvm_alpha = load_rvm_alpha_from_precomputed(testset_root, name, base_alpha_dir_name)
        if rvm_alpha is None:
            print(f"   Precomputed base_alpha not found: {name}")
            return None
        if rvm_alpha.shape[:2] != image.shape[:2]:
            rvm_alpha = cv2.resize(rvm_alpha, (image.shape[1], image.shape[0]))
        print(f"   Using precomputed base_alpha")
    else:
        if rvm_model is None:
            print(f"   RVM model not loaded")
            return None
        rvm_alpha = rvm_model.predict(image)
        print(f"   Running real-time RVM prediction")

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(image_rgb, (512, 512))
    base_resized = cv2.resize(rvm_alpha, (512, 512))

    img_tensor = torch.from_numpy(img_resized).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    base_tensor = torch.from_numpy(base_resized).unsqueeze(0).unsqueeze(0).float()

    img_tensor = img_tensor.to(device)
    base_tensor = base_tensor.to(device)

    with torch.no_grad():
        semantic_alpha = your_model(img_tensor, base_tensor)
        semantic_alpha = semantic_alpha.squeeze().cpu().numpy()
        semantic_alpha = cv2.resize(semantic_alpha, (image.shape[1], image.shape[0]))
        semantic_alpha = np.clip(semantic_alpha, 0, 1)

    refined_alpha = refine_alpha(rvm_alpha, semantic_alpha, delta=delta)

    from skimage.metrics import mean_squared_error
    rvm_mse = mean_squared_error(gt, rvm_alpha)
    refined_mse = mean_squared_error(gt, refined_alpha)
    rvm_sad = np.sum(np.abs(rvm_alpha - gt))
    refined_sad = np.sum(np.abs(refined_alpha - gt))

    print(f"   RVM MSE: {rvm_mse:.6f}, SAD: {rvm_sad:.2f}")
    print(f"   Refined MSE: {refined_mse:.6f}, SAD: {refined_sad:.2f}")
    print(f"   improvement: MSE↓{rvm_mse - refined_mse:.6f} ({((rvm_mse - refined_mse)/rvm_mse*100):.2f}%), SAD↓{rvm_sad - refined_sad:.2f}")

    rvm_composite = composite_with_green_bg(image, rvm_alpha)
    semantic_composite = composite_with_green_bg(image, semantic_alpha)
    refined_composite = composite_with_green_bg(image, refined_alpha)
    original_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(2, 4, figsize=(24, 12))
    fig.suptitle(f"Sample: {name}  |  RVM MSE: {rvm_mse:.6f}  →  Refined MSE: {refined_mse:.6f}  |  Improvement: {(rvm_mse - refined_mse)/rvm_mse*100:.2f}%",
                 fontsize=14, fontweight='bold')

    axes[0, 0].imshow(gt, cmap='gray', vmin=0, vmax=1)
    axes[0, 0].set_title("Ground Truth Alpha", fontsize=12)
    axes[0, 0].axis('off')

    axes[0, 1].imshow(rvm_alpha, cmap='gray', vmin=0, vmax=1)
    axes[0, 1].set_title(f"RVM Alpha\n(MSE: {rvm_mse:.6f})", fontsize=12)
    axes[0, 1].axis('off')

    axes[0, 2].imshow(semantic_alpha, cmap='gray', vmin=0, vmax=1)
    axes[0, 2].set_title("Semantic Alpha", fontsize=12)
    axes[0, 2].axis('off')

    axes[0, 3].imshow(refined_alpha, cmap='gray', vmin=0, vmax=1)
    axes[0, 3].set_title(f"Refined Alpha (δ={delta})\n(MSE: {refined_mse:.6f})", fontsize=12)
    axes[0, 3].axis('off')

    axes[1, 0].imshow(original_rgb)
    axes[1, 0].set_title("Original Image", fontsize=12)
    axes[1, 0].axis('off')

    axes[1, 1].imshow(rvm_composite)
    axes[1, 1].set_title("RVM + Green BG", fontsize=12)
    axes[1, 1].axis('off')

    axes[1, 2].imshow(semantic_composite)
    axes[1, 2].set_title("Semantic + Green BG", fontsize=12)
    axes[1, 2].axis('off')

    axes[1, 3].imshow(refined_composite)
    axes[1, 3].set_title("Refined + Green BG", fontsize=12)
    axes[1, 3].axis('off')

    plt.tight_layout()

    if save_path:
        save_file = os.path.join(save_path, f"comparison_{name}.png")
        plt.savefig(save_file, dpi=150, bbox_inches='tight')
        print(f"   Saved: {save_file}")

    plt.show()
    plt.close(fig)

    return {
        'name': name,
        'rvm_mse': rvm_mse,
        'rvm_sad': rvm_sad,
        'refined_mse': refined_mse,
        'refined_sad': refined_sad,
        'improvement_mse': rvm_mse - refined_mse,
        'improvement_percent': (rvm_mse - refined_mse) / rvm_mse * 100 if rvm_mse > 0 else 0
    }


def show_best_improvements_full(df, testset_root, your_model, device,
                                 delta=0.7, num_samples=8, save_path=None,
                                 use_precomputed=True, rvm_model=None,
                                 base_alpha_dir_name="base_alpha"):

    best_df = df.nlargest(num_samples, 'improvement_mse')

    image_dir = os.path.join(testset_root, 'original_image')
    if not os.path.exists(image_dir):
        image_dir = os.path.join(testset_root, 'blurred_image')
    mask_dir = os.path.join(testset_root, 'mask')

    for idx, (_, row) in enumerate(best_df.iterrows()):
        name = row['name']
        print(f"\nProcessing image {idx+1}/{num_samples}: {name}")

        img_path = os.path.join(image_dir, f"{name}.jpg")
        if not os.path.exists(img_path):
            img_path = os.path.join(image_dir, f"{name}.png")
        mask_path = os.path.join(mask_dir, f"{name}.png")

        image = cv2.imread(img_path)
        gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) / 255.0

        if image is None or gt is None:
            print(f"   Unable to read image: {name}")
            continue

        if use_precomputed:
            rvm_alpha = load_rvm_alpha_from_precomputed(testset_root, name, base_alpha_dir_name)
            if rvm_alpha is None:
                print(f"  Precomputed base_alpha not found: {name}")
                continue
            if rvm_alpha.shape[:2] != image.shape[:2]:
                rvm_alpha = cv2.resize(rvm_alpha, (image.shape[1], image.shape[0]))
        else:
            if rvm_model is None:
                print(f"   RVM model not loaded")
                continue
            rvm_alpha = rvm_model.predict(image)

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(image_rgb, (512, 512))
        base_resized = cv2.resize(rvm_alpha, (512, 512))

        img_tensor = torch.from_numpy(img_resized).permute(2, 0, 1).unsqueeze(0).float() / 255.0
        base_tensor = torch.from_numpy(base_resized).unsqueeze(0).unsqueeze(0).float()

        img_tensor = img_tensor.to(device)
        base_tensor = base_tensor.to(device)

        with torch.no_grad():
            semantic_alpha = your_model(img_tensor, base_tensor)
            semantic_alpha = semantic_alpha.squeeze().cpu().numpy()
            semantic_alpha = cv2.resize(semantic_alpha, (image.shape[1], image.shape[0]))
            semantic_alpha = np.clip(semantic_alpha, 0, 1)

        refined_alpha = refine_alpha(rvm_alpha, semantic_alpha, delta=delta)

        rvm_composite = composite_with_green_bg(image, rvm_alpha)
        semantic_composite = composite_with_green_bg(image, semantic_alpha)
        refined_composite = composite_with_green_bg(image, refined_alpha)
        original_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(2, 4, figsize=(24, 12))
        fig.suptitle(f"Sample: {name}  |  RVM MSE: {row['rvm_mse']:.4f}  →  Refined MSE: {row['refined_mse']:.4f}",
                     fontsize=14, fontweight='bold')

        axes[0, 0].imshow(gt, cmap='gray', vmin=0, vmax=1)
        axes[0, 0].set_title("Ground Truth Alpha", fontsize=12)
        axes[0, 0].axis('off')

        axes[0, 1].imshow(rvm_alpha, cmap='gray', vmin=0, vmax=1)
        axes[0, 1].set_title(f"RVM Alpha\n(MSE: {row['rvm_mse']:.4f})", fontsize=12)
        axes[0, 1].axis('off')

        axes[0, 2].imshow(semantic_alpha, cmap='gray', vmin=0, vmax=1)
        axes[0, 2].set_title("Semantic Alpha", fontsize=12)
        axes[0, 2].axis('off')

        axes[0, 3].imshow(refined_alpha, cmap='gray', vmin=0, vmax=1)
        axes[0, 3].set_title(f"Refined Alpha (δ={delta})\n(MSE: {row['refined_mse']:.4f})", fontsize=12)
        axes[0, 3].axis('off')

        axes[1, 0].imshow(original_rgb)
        axes[1, 0].set_title("Original Image", fontsize=12)
        axes[1, 0].axis('off')

        axes[1, 1].imshow(rvm_composite)
        axes[1, 1].set_title("RVM + Green BG", fontsize=12)
        axes[1, 1].axis('off')

        axes[1, 2].imshow(semantic_composite)
        axes[1, 2].set_title("Semantic + Green BG", fontsize=12)
        axes[1, 2].axis('off')

        axes[1, 3].imshow(refined_composite)
        axes[1, 3].set_title("Refined + Green BG", fontsize=12)
        axes[1, 3].axis('off')

        plt.tight_layout()

        if save_path:
            save_file = os.path.join(save_path, f"comparison_{name}.png")
            plt.savefig(save_file, dpi=150, bbox_inches='tight')
            print(f" Saved: {save_file}")

        plt.show()
        plt.close(fig)


SAVE_DIR = "/content/drive/MyDrive/RVM/comparison_results"
os.makedirs(SAVE_DIR, exist_ok=True)

MANUAL_MODE = False

SELECTED_NAMES = [
    "p_0de2c8bd",
    "p_6aaed593",
    "p_7e060171",
    "p_73aabf07",
    "p_80d05b01",
    "p_87564383",
    "p_a0fe2077",
    "p_a293d5dc",
    "p_bd36e971",
    "p_d3072853",
    "p_dd3fab08",
    "p_8910e0b6",
    "p_e0429404",
    "p_faa76994",
    "p_ffc0b723",
]

try:
    if MANUAL_MODE:
        print("=" * 60)
        print("Manual Mode: Evaluating specified images")
        print("=" * 60)
        print(f"Testset Root: {TESTSET_ROOT}")
        print(f"Selected Images: {SELECTED_NAMES}")
        print("=" * 60)

        results_list = []
        for name in SELECTED_NAMES:
            result = evaluate_single_image(
                name=name,
                testset_root=TESTSET_ROOT,
                your_model=your_model,
                device=DEVICE,
                delta=DELTA,
                use_precomputed=USE_PRECOMPUTED_BASE_ALPHA,
                rvm_model=rvm_model,
                base_alpha_dir_name="base_alpha",
                save_path=SAVE_DIR
            )
            if result:
                results_list.append(result)

        if results_list:
            print("\n" + "=" * 60)
            print("Evaluation Summary for Specified Images")
            print("=" * 60)
            for r in results_list:
                print(f"   {r['name']}: MSE ↓{r['improvement_mse']:.6f} ({r['improvement_percent']:.2f}%)")

    else:

        csv_path = "/content/drive/MyDrive/RVM/full_evaluation_results2.csv"
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            print(f"Loaded CSV: {csv_path}")

            show_best_improvements_full(
                df=df,
                testset_root=TESTSET_ROOT,
                your_model=your_model,
                device=DEVICE,
                delta=DELTA,
                num_samples=8,
                save_path=SAVE_DIR,
                use_precomputed=USE_PRECOMPUTED_BASE_ALPHA,
                rvm_model=rvm_model,
                base_alpha_dir_name="base_alpha"
            )
        else:
            print("Evaluation results CSV not found. Please run main() first.")

except NameError as e:
    print(f"Error: {e}")